In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from models.timefm import TimesFMModel
from models.lstm import LSTMModel, QLSTMModel
from models.gan import QGanModel, MultiSequenceGAN
from sklearn.preprocessing import StandardScaler

stocks = ['nifty', 'tcs', 'hindalco', 'jswsteel']

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm


# ============================================================
# Store results for all stocks
# ============================================================
all_results = []


# ============================================================
# Progress bar over stocks
# ============================================================
for stock in tqdm(stocks, desc="Training MultiSequenceGAN-QC", unit="stock"):

    print(f"\n{'='*80}")
    print(f"Processing: {stock}")
    print(f"{'='*80}")

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------
    data = np.load(f"data/{stock}_l10y.npy")

    # Flatten in case data is (N, 1)
    original_data = np.asarray(data).reshape(-1)

    # --------------------------------------------------------
    # Original data statistics
    # --------------------------------------------------------
    data_info = {
        "Stock": stock,
        "N": len(original_data),
        "Mean": np.mean(original_data),
        "Std": np.std(original_data),
        "Variance": np.var(original_data),
        "Min": np.min(original_data),
        "Max": np.max(original_data),
    }

    print("\nOriginal data statistics:")
    print(
        pd.DataFrame([data_info]).to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    # ========================================================
    # Model
    # ========================================================
    msgan_qc = MultiSequenceGAN(
        data=data,
        type="QC",
        historical_lookup=30,
        horizon=15,
        lag=1,
        latent_size=8,
        n_qubits=6,
        quantum_layers=4,
        hidden_size=32,
        epochs=100,
        batch_size=32,
        learning_rate=1e-4,
        diversity_loss_weight=0.5,
        variety_loss_weight=1.0,
        train_ratio=0.8,
        seed=42,
    )

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    print("\nTraining...")

    history = msgan_qc.train()

    # --------------------------------------------------------
    # Backtest
    # --------------------------------------------------------
    print("Running backtest...")

    d = msgan_qc.backtest(return_all_sequences=True)

    contexts = d["contexts"]
    predictions = d["predictions"]
    actuals = d["actuals"]
    all_seq = d["all_sequences"]

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    print("Calculating metrics...")

    prob_result = msgan_qc.probabilistic_metrics(
        all_seq,
        actuals
    )

    result = msgan_qc.metrics(
        predictions,
        actuals
    )

    # ========================================================
    # Basic prediction statistics
    # ========================================================

    actual_flat = actuals.reshape(-1)
    pred_flat = predictions.reshape(-1)

    original_mean = np.mean(original_data)
    original_std = np.std(original_data)
    original_var = np.var(original_data)

    actual_mean = np.mean(actual_flat)
    actual_std = np.std(actual_flat)
    actual_var = np.var(actual_flat)

    pred_mean = np.mean(pred_flat)
    pred_std = np.std(pred_flat)
    pred_var = np.var(pred_flat)

    # --------------------------------------------------------
    # Variance preservation
    # --------------------------------------------------------
    variance_ratio = pred_var / (actual_var + 1e-12)

    variance_error_pct = (
        abs(pred_var - actual_var)
        / (actual_var + 1e-12)
        * 100
    )

    # --------------------------------------------------------
    # Mean / standard deviation errors
    # --------------------------------------------------------
    mean_error = pred_mean - actual_mean

    std_error = pred_std - actual_std

    mean_error_pct = (
        abs(mean_error)
        / (abs(actual_mean) + 1e-12)
        * 100
    )

    std_error_pct = (
        abs(std_error)
        / (actual_std + 1e-12)
        * 100
    )

    # ========================================================
    # Correlation
    # ========================================================
    correlation = np.corrcoef(
        actual_flat,
        pred_flat
    )[0, 1]

    # ========================================================
    # Directional accuracy
    #
    # Compare whether predicted movement has the same
    # direction as actual movement.
    # ========================================================
    actual_diff = np.diff(actual_flat)
    pred_diff = np.diff(pred_flat)

    directional_accuracy = np.mean(
        np.sign(actual_diff) == np.sign(pred_diff)
    ) * 100

    # ========================================================
    # Standard forecasting metrics
    # ========================================================
    overall = result["overall"]

    mae = overall.get("mae", np.nan)
    rmse = overall.get("rmse", np.nan)
    mape = overall.get("mape", np.nan)
    r2 = overall.get("r2", np.nan)

    # ========================================================
    # Collect everything
    # ========================================================
    stock_result = {
        "Stock": stock,

        # Dataset
        "N": len(original_data),

        # Original
        "Original Mean": original_mean,
        "Original Std": original_std,
        "Original Variance": original_var,

        # Test actual
        "Actual Mean": actual_mean,
        "Actual Std": actual_std,
        "Actual Variance": actual_var,

        # Prediction
        "Pred Mean": pred_mean,
        "Pred Std": pred_std,
        "Pred Variance": pred_var,

        # Distribution preservation
        "Variance Ratio": variance_ratio,
        "Variance Error %": variance_error_pct,
        "Mean Error": mean_error,
        "Mean Error %": mean_error_pct,
        "Std Error": std_error,
        "Std Error %": std_error_pct,

        # Forecasting metrics
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2,

        # Dependency / direction
        "Correlation": correlation,
        "Directional Accuracy %": directional_accuracy,
    }

    all_results.append(stock_result)

    # ========================================================
    # Print current stock summary
    # ========================================================
    summary = pd.DataFrame([stock_result])

    cols_to_show = [
        "Stock",
        "Original Variance",
        "Actual Variance",
        "Pred Variance",
        "Variance Ratio",
        "MAE",
        "RMSE",
        "MAPE",
        "R2",
        "Correlation",
        "Directional Accuracy %",
    ]

    print("\nPerformance:")
    print(
        summary[cols_to_show].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

    # ========================================================
    # 1. Day +1 forecast
    # ========================================================
    day1_actual = actuals[:, 0]
    day1_pred = predictions[:, 0]

    plt.figure(figsize=(12, 5))

    plt.plot(
        day1_actual,
        label="Actual (Day +1)",
        color="black"
    )

    plt.plot(
        day1_pred,
        label="Predicted (Day +1)",
        color="tab:red",
        linestyle="--"
    )

    plt.title(
        f"{stock} MultiSequenceGAN [{msgan_qc.type}] "
        "— Day+1 Forecast vs Actual (Test Set)"
    )

    plt.xlabel("Test window index")
    plt.ylabel("Value")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ========================================================
    # 2. Full future window
    # ========================================================
    idx = 0

    ensemble = all_seq[idx]

    x_context = np.arange(
        -msgan_qc.historical_lookup,
        0
    )

    x_future = np.arange(
        0,
        msgan_qc.horizon
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        x_context,
        contexts[idx],
        label="Context (history)",
        color="gray"
    )

    plt.plot(
        x_future,
        actuals[idx],
        label="Actual future",
        color="black",
        marker="o"
    )

    plt.plot(
        x_future,
        predictions[idx],
        label="Predicted (best trajectory)",
        color="tab:red",
        linestyle="--",
        marker="x"
    )

    for s in range(ensemble.shape[1]):
        plt.plot(
            x_future,
            ensemble[:, s],
            color="tab:red",
            alpha=0.05
        )

    plt.axvline(
        0,
        color="gray",
        linestyle=":",
        linewidth=1
    )

    plt.title(
        f"Test window {idx} for {stock}: "
        "context, actual vs predicted future"
    )

    plt.xlabel("Days relative to forecast origin")
    plt.ylabel("Value")
    plt.legend()
    plt.tight_layout()
    plt.show()


# ============================================================
# Final comparison table
# ============================================================
results_df = pd.DataFrame(all_results)

print("\n")
print("=" * 120)
print("FINAL MULTI-SEQUENCE GAN — QC RESULTS")
print("=" * 120)

display_cols = [
    "Stock",
    "Original Variance",
    "Actual Variance",
    "Pred Variance",
    "Variance Ratio",
    "MAE",
    "RMSE",
    "MAPE",
    "R2",
    "Correlation",
    "Directional Accuracy %",
]

print(
    results_df[display_cols].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

In [ ]:
prob_result

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1) Day+1-ahead forecast vs actual, across the whole chronological test set
day1_actual = actuals[:, 0]
day1_pred = predictions[:, 0]

plt.figure(figsize=(12, 5))
plt.plot(day1_actual, label="Actual (Day +1)", color="black")
plt.plot(day1_pred, label="Predicted (Day +1)", color="tab:red", linestyle="--")
plt.title(f"MultiSequenceGAN [{msgan_qc.type}] — Day+1 Forecast vs Actual (Test Set)")
plt.xlabel("Test window index")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()

# 2) Full 6-day window for one test example: context, actual future, predicted
#    future, and the full generated ensemble spread
idx = 0  # change to inspect a different test window
ensemble = all_seq[idx]  # shape: (horizon, n_sequences)

x_context = np.arange(-msgan_qc.historical_lookup, 0)
x_future = np.arange(0, msgan_qc.horizon)

plt.figure(figsize=(8, 5))
plt.plot(x_context, contexts[idx], label="Context (history)", color="gray")
plt.plot(x_future, actuals[idx], label="Actual future", color="black", marker="o")
plt.plot(x_future, predictions[idx], label="Predicted (best trajectory)", color="tab:red", linestyle="--", marker="x")
for s in range(ensemble.shape[1]):
    plt.plot(x_future, ensemble[:, s], color="tab:red", alpha=0.05)  # faint ensemble spread
plt.axvline(0, color="gray", linestyle=":", linewidth=1)
plt.title(f"Test window {idx}: context, actual vs predicted future")
plt.xlabel("Days relative to forecast origin")
plt.ylabel("Value")
plt.legend()
plt.tight_layout()
plt.show()